# StoreDNA - Pipeline stages

Data flows **top to bottom**. This notebook implements **Stage 6** (highlighted).

```
│  1. Source Data          │
│  2. Curate Data          │
│  3. AI Enrichment        │
│  4. Modality Vectors     │
│  5. StoreDNA Builder     │
│  6. Vector Index    ◀── Current Step │
│  7. Business Output      │
```

| Stage | Name | Status |
|-------|------|--------|
| 1 | Source Data | Complete |
| 2 | Curate Data | Complete |
| 3 | AI Enrichment | Complete |
| 4 | Modality Vectors | Complete |
| 5 | StoreDNA Builder | Complete |
| **6** | **Vector Index** | **Current** |
| 7 | Business Output | Next |


# Retail Store DNA Builder - Stage 6: Vector Index

## What is Stage 6?

Stage 6 creates a **new Azure AI Search index** for the Stage 5 `store_dna_vector` outputs and uploads **one document per store**.

Because Azure AI Search supports vector dimensions up to **4096**, this stage first creates a **search-friendly projected vector** from the Stage 5 `store_dna_vector`, then uploads that projected vector to a separate store-level index.

This notebook uses the Azure AI Search credentials from `.env` and creates a separate index for store-level vectors.

---

## Inputs & outputs

| Input | Source | Stage 6 action | Output |
|-------|--------|----------------|--------|
| `store_dna_vectors.npz` | Stage 5 | Project to search-safe vector + upload | Azure AI Search store index |
| `store_dna_index.csv` | Stage 5 | Add QC metadata | Search documents |
| `dim_store.csv` | Stage 2 | Add store metadata | Search documents |

**Local output directory:** `data/USA_100_Stores/vector_index/`

**Default new index name:** `store-dna-store-vectors`
**Default search vector dimension:** `2048`


## 1. Setup

In [ ]:
%pip install -q -r ../requirements.txt

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.store_dna_vector_index import (
    StoreVectorIndexConfig,
    get_store_vector_index_count,
    load_store_dna_outputs,
    run_store_vector_index,
)

load_dotenv(PROJECT_ROOT / ".env")

CURATED_DIR = PROJECT_ROOT / "data" / "USA_100_Stores" / "curated"
STORE_DNA_DIR = PROJECT_ROOT / "data" / "USA_100_Stores" / "store_dna"
VECTOR_INDEX_DIR = PROJECT_ROOT / "data" / "USA_100_Stores" / "vector_index"

print("Project root:", PROJECT_ROOT)
print("Curated input:", CURATED_DIR)
print("Stage 5 input:", STORE_DNA_DIR)
print("Stage 6 output:", VECTOR_INDEX_DIR)


## 2. Configuration

In [ ]:
NEW_INDEX_NAME = "store-dna-store-vectors"
SEARCH_VECTOR_DIMENSIONS = 2048

config = StoreVectorIndexConfig.from_env(
    PROJECT_ROOT,
    index_name=NEW_INDEX_NAME,
    vector_dimensions=SEARCH_VECTOR_DIMENSIONS,
)

print("Search endpoint:", config.search_endpoint)
print("New index name:", config.index_name)
print("Upload batch size:", config.upload_batch_size)
print("Search vector dimensions:", config.vector_dimensions)


## 3. Preview Stage 5 inputs

In [ ]:
store_ids, vectors, index_df = load_store_dna_outputs(STORE_DNA_DIR)
print("Stores:", len(store_ids))
print("Vector shape:", vectors.shape)
index_df.head()


## 4. Run full Stage 6 - create new index and upload store vectors

In [ ]:
manifest = run_store_vector_index(CURATED_DIR, STORE_DNA_DIR, VECTOR_INDEX_DIR, config)
print(json.dumps(manifest, indent=2))


## 5. Verify uploaded document count

In [ ]:
doc_count = get_store_vector_index_count(config)
print("Index document count:", doc_count)


## 6. Local output layout

```
data/USA_100_Stores/vector_index/
├── store_dna_search_vectors.npz
└── store_vector_index_manifest.json
```

---

## 7. Next steps - Stage 7 (Business Output)

| Action | Description |
|--------|-------------|
| Peer lookup | Query similar stores using vector search |
| Clustering | Group stores by StoreDNA similarity |
| Narratives | Generate store summaries from nearest neighbors |
